$$\large{\color{yellow}{\text{MDS6304 Deep Learning ETEPW}}}$$

$$\large\text{Theme}: \underline{\text{building a softmax classifier from scratch using efficient computations in PyTorch}}$$

---

We consider a softmax classfier, which is a 1-layer neural network or a 0-hidden layer neural network, for a batch comprising $b$ samples represented as the $b\times 784$-matrix $$\mathbf{X} = \begin{bmatrix}{\mathbf{x}^{(0)}}^\mathrm{T}\\{\mathbf{x}^{(1)}}^\mathrm{T}\\\vdots\\{\mathbf{x}^{(b-1)}}^\mathrm{T}\end{bmatrix}$$ with one-hot encoded true labels represented as the $b\times 10$-matrix (10 possible categories) $$\mathbf{Y}=\begin{bmatrix}{\mathbf{y}^{(0)}}^\mathrm{T}\\{\mathbf{y}^{(1)}}^\mathrm{T}\\\vdots\\{\mathbf{y}^{(b-1)}}^\mathrm{T}\end{bmatrix}.$$

The forward propagation for a generic sample in the batch seen as a $1\times784$-object $\mathbf{x}^\mathrm{T}$ with the bias feature $1$ added is presented below:

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}_B^\mathrm{T}}_{1\times785}=\begin{bmatrix}\mathbf{x}^\mathrm{T}&1\end{bmatrix}}&\rightarrow\boxed{\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times 10} = \underbrace{\mathbf{x}_B^\mathrm{T}}_{1\times785}\underbrace{{\mathbf{W}}}_{785\times10}}\rightarrow\boxed{\underbrace{{\mathbf{a}}^\mathrm{T}}_{1\times10}=\text{softmax}\left(\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times10}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^9-y_k\log\left(\hat{y}_k\right)}.
\end{align*}$$

The forward propagation for the same generic sample seen as a $784$-vector $\mathbf{x}$ with the bias feature $1$ added is presented below (note that the weight matrix has the same name $\mathbf{W}$ as above for simplicity even though it should show up as $\mathbf{W}^\mathrm{T}$):

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}_B}_{785}=\begin{bmatrix}\mathbf{x}\\1\end{bmatrix}}&\rightarrow\boxed{\underbrace{\mathbf{z}}_{10} = \underbrace{\mathbf{W}}_{10\times785}\underbrace{\mathbf{x}_B}_{785}}\rightarrow\boxed{\underbrace{\mathbf{a}}_{10}=\text{softmax}\left(\underbrace{\mathbf{z}}_{10}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^9-y_k\log\left(\hat{y}_k\right)}.\end{align*}$$

We will derive the update rule for the weights matrix $\mathbf{W}$ using the setup above.


The average crossentropy (CCE) loss for the batch is:$$\begin{align*}L &=\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]\\&=\frac{1}{b}\left[\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(0)}\log\left(\hat{y}^{(0)}_k\right)+\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(1)}\log\left(\hat{y}^{(1)}_k\right)+\cdots+\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(b-1)}\log\left(\hat{y}^{(b-1)}_k\right)\right]\\&=\frac{1}{b}\left[{\color{yellow}-}{\mathbf{y}^{(0)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(0)}}\right)+{\color{yellow}-}{\mathbf{y}^{(1)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(1)}}\right)+\cdots+{\color{yellow}-}{\mathbf{y}^{(b-1)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(b-1)}}\right)\right].\end{align*}$$

The computational graph for the samples, each at a time treated as a $785$-vector, in the batch are presented below where the weights matrix has shape $10\times 785.$

$\hspace{1.5in}\begin{align*}L_0\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(0)} &= \mathbf{a}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}$$\hspace{0.25in}\begin{align*}L_1\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(1)} &= \mathbf{a}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}$$\qquad\cdots\qquad$$\begin{align*} L_{b-1}\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(b-1)} &= \mathbf{a}^{(b-1)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(b-1)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}$

The gradient of the average batch loss w.r.t. the weights is:
$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &=\frac{1}{b}\left[\nabla_\mathbf{W}(L_0)+\nabla_\mathbf{W}(L_1)+\cdots+\nabla_\mathbf{W}(L_{b-1})\right]\end{align*}$$
which by chain rule can be written as:

$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &= \frac{1}{b}\left(\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(0)}\right) \times\nabla_{\mathbf{z}^{(0)}}\left(\hat{\mathbf{y}}^{(0)}\right)\times\nabla_{\hat{\mathbf{y}}^{(0)}}(L_0)\right]}_{\nabla_\mathbf{W}(L_0)}+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(1)}\right) \times\nabla_{\mathbf{z}^{(1)}}\left(\hat{\mathbf{y}}^{(1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(1)}}(L_1)\right]}_{\nabla_\mathbf{W}(L_1)}+\cdots+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(b-1)}\right) \times\nabla_{\mathbf{z}^{(b-1)}}\left(\hat{\mathbf{y}}^{(b-1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(b-1)}}(L_{b-1})\right]}_{\nabla_\mathbf{W}(L_{b-1})}\right)\\&=\frac{1}{b}\left(\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(0)}\right) \times\nabla_{\mathbf{z}^{(0)}}\left({\mathbf{a}}^{(0)}\right)\times\nabla_{\hat{\mathbf{y}}^{(0)}}(L_0)\right]}_{\nabla_\mathbf{W}(L_0)}+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(1)}\right) \times\nabla_{\mathbf{z}^{(1)}}\left({\mathbf{a}}^{(1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(1)}}(L_1)\right]}_{\nabla_\mathbf{W}(L_1)}+\cdots+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(b-1)}\right) \times\nabla_{\mathbf{z}^{(b-1)}}\left(\hat{\mathbf{y}}^{(b-1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(b-1)}}(L_{b-1})\right]}_{\nabla_\mathbf{W}(L_{b-1})}\right)\\&=\frac{1}{b}\sum_{i=0}^{b-1}\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right) \times\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)\right]\\&=\frac{1}{b}\sum_{i=0}^{b-1}\left[\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}_B}\right) \times\nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right)\right],\end{align*}$$
which can be written as

$$\begin{align*}\nabla_{\mathbf{W}}(L) &= \dfrac{1}{b}\displaystyle\sum_{i=0}^{b-1}\underbrace{\begin{bmatrix}\boxed{{\mathbf{x}^{(i)}_B}\ \pmb{0}\ \pmb{0}\ \ldots\ \pmb{0}}&&&&\\\\&\boxed{\pmb{0}\ {\mathbf{x}^{(i)}_B}\ \pmb{0}\ \ldots\ \pmb{0}}&&&\\&\hspace{1cm}\ddots&&&\\&&\hspace{-0.5cm}\ddots&&\\&&&\boxed{\pmb{0}\ \pmb{0}\ \ldots\ \pmb{0}\ {\mathbf{x}^{(i)}_B}}&\end{bmatrix}}_{\color{cyan}{\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right)=\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}_B}\right):\ 10\times785\times10}}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & \cdots & -a^{(i)}_0 a^{(i)}_9\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & \cdots & -a^{(i)}_1 a^{(i)}_9\\\vdots & \vdots & \ddots & \vdots\\-a^{(i)}_9 a^{(i)}_0 & -a^{(i)}_9 a^{(i)}_1 & \cdots & a^{(i)}_9 (1 - a^{(i)}_9)\end{bmatrix}}_{\color{cyan}{\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right) = \nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right):\ 10\times10}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0}\\-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\\vdots\\-\frac{y^{(i)}_9}{\hat{y}^{(i)}_9}\end{bmatrix}}_{\color{cyan}{\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)=\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right):\ 10\times1}}\end{align*}$$

The forward and backward propagation showing the gradient flow for a generic sample is shown below:

![](https://1drv.ms/i/c/37720f927b6ddc34/IQS3b-biQ4W9QpCtJzaZnyCoAQ8_r9i707rpOE1O9I0yntM?width=686&height=93)

$$\begin{align*}\nabla_{\mathbf{W}}(L) &=\dfrac{1}{b}\displaystyle\sum_{i=0}^{b-1}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & \cdots & -a^{(i)}_0 a^{(i)}_9\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & \cdots & -a^{(i)}_1 a^{(i)}_9\\\vdots & \vdots & \ddots & \vdots\\-a^{(i)}_9 a^{(i)}_0 & -a^{(i)}_9 a^{(i)}_1 & \cdots & a^{(i)}_9 (1 - a^{(i)}_9)\end{bmatrix}}_{\color{cyan}{=\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0} \\
-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\\vdots\\-\frac{y^{(i)}_9}{\hat{y}^{(i)}_9}\end{bmatrix}}_{\color{cyan}{=-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}}}{\mathbf{x}^{(i)}_B}^\mathrm{T}.\end{align*}$$

We can write the gradient in the following way for efficient coding purposes: $$\nabla_\mathbf{W}(L) = \frac{1}{b}\sum_{i=0}^{b-1}\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}\right]\left[-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}\right]{\mathbf{x}^{(i)}_B}^\mathrm{T}.$$


It can be seen that the gradient object has shape $(10\times 10)\times(10\times1)\times(1\times785)=10\times785,$ which is the same shape as the weights matrix $\mathbf{W}.$ However, our derivation here assumed that the samples are seen as column vectors of the data matrix. The original data matrix has the samples arranged as rows which corresponded to the weights matrix of shape $785\times10.$ In order to get the gradient w.r.t. that weights matrix, we have to transpose this expression resulting in the update $$\nabla_\mathbf{W}(L) = \frac{1}{b}\sum_{i=0}^{b-1}\underbrace{\mathbf{x}^{(i)}_B}_{\color{yellow}{785\times1}}\underbrace{\underbrace{\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]}_{\color{magenta}{\text{output side gradient of softmax layer: }1\times10}}\underbrace{\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\color{magenta}{\text{local gradient of softmax layer: }10\times10}}}_{\color{yellow}{\text{output side gradient of dense layer: }1\times10}}.$$



Note that when regularization is applied to the weights using a regularization strength $\lambda,$ then the regularization loss gets added to the data loss to give the total loss for the batch as follows: $$\begin{align*}L &= L_\text{data}+L_\text{reg}\\&=\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]+\lambda\left[{\mathbf{w}^{(0)}}^\mathrm{T}\mathbf{w}^{(0)}+{\mathbf{w}^{(1)}}^\mathrm{T}\mathbf{w}^{(1)}+\cdots+{\mathbf{w}^{(783)}}^\mathrm{T}\mathbf{w}^{(783)}\right].\end{align*}$$
Note that the last row of the weights matrix $\mathbf{W},$ which comprises the bias values, are not included in the regularization loss.

The gradient w.r.t. the weights matrix $\mathbf{W}$ now also includes the gradient w.r.t. the regularization loss as follows: $$\begin{align*}\nabla_\mathbf{W}(L)&=\nabla_\mathbf{W}\left(\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]\right)+\lambda\nabla_\mathbf{W}\left(L_\text{reg}\right)\\&=\underbrace{\frac{1}{b}\sum_{i=0}^{b-1}{\mathbf{x}^{(i)}_B}\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\text{data gradient}}+\lambda\underbrace{\begin{bmatrix}2{\mathbf{w}^{(0)}}^\mathrm{T}\\2{\mathbf{w}^{(1)}}^\mathrm{T}\\\vdots\\2{\mathbf{w}^{(783)}}^\mathrm{T}\\\pmb{0}\end{bmatrix}}_{\text{regularization gradient}}.\end{align*}$$


---

---

Load libraries

---

In [ ]:
## Load libraries
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

---

Mount Google drive

---

In [ ]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/Office of Online Education/MDS6304_Webinar_October2025'
    DATA_DIR = DIR + '/Data/'
else:
    DATA_DIR = 'Data/'

---

Load diabetes data (binary & multilabel classification)

---

In [ ]:
## Load diabetes data
file = DATA_DIR+'diabetes.csv'
df= pd.read_csv(file, header = 0).dropna()

## Train and test split of the data
X = df.loc[:, df.columns != 'Outcome']
y = df['Outcome']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

# Standardize data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Convert train and test data to torch tensors (note that Y should be a 1-column matrix)
X_train = torch.tensor(X_train, dtype = torch.float64)
X_test = torch.tensor(X_test, dtype = torch.float64)
# Label encode class labels (do the following if using logistic regression for multilabel classification)
Y_train = torch.tensor(Y_train.values).reshape(-1, 1)
Y_test = torch.tensor(Y_test.values).reshape(-1, 1)
# One-hot encode class labels (do the following if using softmax for multilabel classification)
#Y_train =  nn.functional.one_hot(torch.tensor(Y_train.values, dtype = torch.int64))
#Y_test = nn.functional.one_hot(torch.tensor(Y_test.values, dtype = torch.int64))


num_samples = X_train.shape[0]
num_features = X_train.shape[1]
num_labels = len(np.unique(Y_train))

print('Diabetes data set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))

---

Load MNIST Data (multilabel classification)

---

In [ ]:
## Load MNIST data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = torch.tensor(X_train.reshape(X_train.shape[0], X_train.shape[1]*X_train.shape[2]))
X_test = torch.tensor(X_test.reshape(X_test.shape[0], X_test.shape[1]*X_test.shape[2]))

num_samples = X_train.shape[0]
num_labels = len(np.unique(y_train))
num_features = X_train.shape[1]

# One-hot encode class labels
Y_train =  nn.functional.one_hot(torch.tensor(y_train, dtype = torch.int64))
Y_test = nn.functional.one_hot(torch.tensor(y_test, dtype = torch.int64))

# Normalize the samples (images) using the training data
xmax = torch.amax(X_train) # 255
xmin = torch.amin(X_train) # 0
X_train = (X_train - xmin) / (xmax - xmin) # all train features turn into a number between 0 and 1
X_test = (X_test - xmin) / (xmax - xmin)

print('MNIST set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))
print('Number of output labels = %d'%(num_labels))

---

A generic layer class with forward and backward methods

----

In [ ]:
class Layer(nn.Module):
  def __init__(self):
    super(Layer, self).__init__()
    ?

  def forward(self, input):
    raise NotImplementedError("Forward propagation not implemented")

  def backward(self, output_gradient, learning_rate):
    raise NotImplementedError("Backward propagation not implemented")

---

Categorical crossentropy (CCE) loss and its gradient for the batch samples (for multilabel classification)

---

In [ ]:
## Define the loss function and its gradient
class CategoricalCrossEntropyLoss:
  def forward(self, Y, Yhat):
    return t?

  def backward(self, Y, Yhat):
    return ?

---

Softmax activation layer class


---

In [ ]:
## Softmax activation layer class
class Softmax(Layer):
  def forward(self, input):
    self.input = ?
    exp_values = torch.exp(self.input - torch.max(self.input, dim = -1, keepdim = True).values)
    self.output = exp_values / torch.sum(exp_values, dim = 1, keepdim = True)
    # self.output = F.softmax(self.input, dim = -1)
    return self.output

  def backward(self, output_gradient, learning_rate = None):
    I = ?
    softmax_local_gradient = ?
    # Calculate gradient flowing back on the input side of the softmax layer
    input_gradient = ?
    return input_gradient

---

Dense layer class

---

In [ ]:
## Dense layer class
class Dense(Layer):
  def __init__(self, input_size, output_size):
    super(Dense, self).__init__()
    self.weights = ?
    with torch.no_grad():
      self.weights[?].fill_(0.01)  # set all bias values to the same nonzero constant

  def forward(self, input):
    self.input = ? # bias trick
    self.output= ?
    return self.output

  def backward(self, output_gradient, learning_rate):
    # Calculate local gradient w.r.t. weights
    dense_local_weights_gradient = (1/output_gradient.shape[0]) * ?
    # Update weights for dense layer
    with torch.no_grad():
      self.weights.data += ?

---

Neural network class for multilbel classification (loss function is categorical crossentropy and last layer is softmax-activated).


---

In [ ]:
class NeuralNetwork:
  def __init__(self, num_features, num_labels, loss_fn, learning_rate = 1e-02):
    self.num_features = num_features
    self.num_labels = num_labels
    self.loss_fn = loss_fn
    self.learning_rate = learning_rate
    # Architecture
    self.layers = [
        Dense(?, ?),
        Softmax()
        ]

  # Forward propagation
  def forward(self, x):
    for layer in ?:
      x = ?
    return x

  # Backward propagation
  def backward(self, loss_gradient):
    for layer in reversed(?):
      loss_gradient = layer.backward(?, ?)

---

Train a neural network for multilabel classification (loss function is categorical crossentropy).


---

In [ ]:
# Initialize model and optimizer
learning_rate = 1e-03
nepochs = 100
reg_strength = 0.1
batch_size = 32

# Choose appropriate loss function
loss_fn = CategoricalCrossEntropyLoss()

# Data loader for batch processing
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size = batch_size, shuffle = True)
test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size = X_test.shape[0], shuffle = False)

model = NeuralNetwork(num_features, num_labels, loss_fn, learning_rate)

# Create empty list to store training and test losses over each epoch
train_loss = [None]*nepochs
test_loss = [None]*nepochs

for epoch in range(nepochs):
  loss = 0.0

  for x_batch, y_batch in train_loader:
    # Forward pass
    predictions = ?
    loss = loss + ?

    # Backward pass
    loss_gradient = ?
    model.backward(?)

  train_loss[epoch] = loss.detach().numpy() / len(train_loader)

  # Test loss calculation
  loss = 0.0

  with torch.no_grad():
    for x_batch, y_batch in test_loader:
      predictions = model.forward(x_batch)
      loss = loss + model.loss_fn.forward(y_batch, predictions)

  test_loss[epoch] = loss.detach().numpy()

  print(f"Epoch {epoch + 1}/{nepochs}, Train Loss: {train_loss[epoch]:.4f}, Test Loss: {test_loss[epoch]:.4f}")

---

Plot training and test loss vs. epoch

---

In [ ]:
## Plot train and test loss as a function of epoch:
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
fig.tight_layout(pad = 4.0)
ax.plot(train_loss, 'b', label = 'Train')
ax.plot(test_loss, 'r', label = 'Test')
ax.set_xlabel('Epoch', fontsize = 12)
ax.set_ylabel('Loss value', fontsize = 12)
ax.legend()
ax.set_title('Loss vs. Epoch', fontsize = 14);

---

Assess model performance on test data

---

In [ ]:
## Assess model performance on test data (multilabel classification)
with torch.no_grad():
  for x_batch, y_batch in test_loader:
    predictions = ?

# Predicted labels for multilabel classification
ypred = ?

# True labels for multilabel classification
ytrue = ?

# Classifier performace
print(f'Accuracy on test data = {(torch.mean((ytrue == ypred).float())*100):3.2f}')
# Print confusion matrix
print(confusion_matrix(ytrue, ypred))

In [ ]:
## Plot a random test sample with its predicted label printed above the plot
test_index = np.random.choice(X_test.shape[0])
fig, ax = plt.subplots(1, 1, figsize = (2, 2))
print(f'Image classified as {ypred[test_index]}')
ax.imshow(X_test[test_index].detach().numpy().reshape(28, 28), cmap = 'gray');